# Clustering Models Experiments

This notebook performs comprehensive clustering experiments using K-Means and Hierarchical clustering algorithms.

## Objectives
- Implement and compare K-Means and Hierarchical clustering
- Find optimal number of clusters using multiple methods
- Evaluate clustering performance with comprehensive metrics
- Visualize and interpret clustering results
- Generate business insights from cluster analysis

In [ ]:
# Import necessary libraries
import sys
sys.path.append('../src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

# Import our custom modules
from models.kmeans_model import KMeansModel
from models.hierarchical_model import HierarchicalModel
from evaluation.metrics import ClusteringEvaluator
from config.settings import settings
from config.logging_config import setup_logging, get_logger
from utils.helpers import save_json, load_json

# Set up logging
setup_logging()
logger = get_logger(__name__)

# Set plotting style
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

print("Libraries imported successfully!")

## 1. Load Prepared Data

In [ ]:
# Load the prepared datasets from Phase 3
import os

# Load original features dataset
try:
    X_original = np.load('../data/processed/X_original_features.npy')
    feature_selection_results = load_json('../reports/feature_selection_results.json')
    feature_names = feature_selection_results['final_features']
    print(f"Loaded original features: {X_original.shape}")
    print(f"Feature names: {feature_names}")
except FileNotFoundError:
    print("Processed data not found. Please run Phase 3 first.")
    # For demonstration, create sample data
    from data.loader import DataLoader
    loader = DataLoader()
    df = loader.load_sample_data()
    
    # Simple preprocessing
    from data.preprocessor import DataPreprocessor
    preprocessor = DataPreprocessor()
    df_processed = preprocessor.fit_transform(df)
    
    # Select numeric features
    X_original = df_processed.select_dtypes(include=[np.number]).values
    feature_names = df_processed.select_dtypes(include=[np.number]).columns.tolist()
    print(f"Created sample data: {X_original.shape}")

# Load PCA reduced dataset
try:
    X_pca = np.load('../data/processed/X_pca_95_variance.npy')
    print(f"Loaded PCA reduced data: {X_pca.shape}")
except FileNotFoundError:
    print("PCA data not found, using original features")
    X_pca = X_original

# Load 2D datasets for visualization
try:
    X_pca_2d = np.load('../data/processed/X_pca_2d.npy')
    X_tsne_2d = np.load('../data/processed/X_tsne_2d.npy')
    print(f"Loaded 2D visualization data: PCA {X_pca_2d.shape}, t-SNE {X_tsne_2d.shape}")
except FileNotFoundError:
    print("2D visualization data not found, will create during analysis")
    X_pca_2d = None
    X_tsne_2d = None

print(f"\nData Summary:")
print(f"Original features: {X_original.shape}")
print(f"PCA reduced: {X_pca.shape}")
if X_pca_2d is not None:
    print(f"PCA 2D: {X_pca_2d.shape}")
if X_tsne_2d is not None:
    print(f"t-SNE 2D: {X_tsne_2d.shape}")

## 2. K-Means Clustering Experiments

In [ ]:
# Initialize K-Means model and evaluator
kmeans = KMeansModel()
evaluator = ClusteringEvaluator()

print("=== K-Means Clustering Experiments ===")
print(f"Data shape: {X_original.shape}")
print(f"Feature names: {feature_names[:5]}..." if len(feature_names) > 5 else f"Feature names: {feature_names}")

In [ ]:
# Experiment 1: Find Optimal Number of Clusters
print("=== Experiment 1: Finding Optimal K ===")

# Use original features for finding optimal k
optimal_k = kmeans.find_optimal_k(
    X_original,
    k_range=(2, 12),
    method='both',  # Use both elbow and silhouette
    plot_results=True
)

print(f"Optimal number of clusters: {optimal_k}")

# Display elbow and silhouette data
print("\nElbow method data:")
for k, inertia in zip(kmeans.elbow_data['k_values'], kmeans.elbow_data['inertias']):
    print(f"k={k}: inertia={inertia:.2f}")

print("\nSilhouette analysis data:")
for k, score in zip(kmeans.silhouette_data['k_values'], kmeans.silhouette_data['silhouette_scores']):
    print(f"k={k}: silhouette={score:.4f}")

In [ ]:
# Experiment 2: Fit K-Means with Optimal K
print("=== Experiment 2: K-Means with Optimal K ===")

# Fit K-Means with optimal number of clusters
kmeans.fit(X_original, n_clusters=optimal_k)

# Get model summary
kmeans_summary = kmeans.get_model_summary()
print(f"\nK-Means Model Summary:")
for key, value in kmeans_summary.items():
    if key not in ['elbow_data', 'silhouette_data', 'cluster_centers']:
        print(f"{key}: {value}")

# Display cluster distribution
print(f"\nCluster Distribution:")
unique_labels, counts = np.unique(kmeans.cluster_labels, return_counts=True)
for label, count in zip(unique_labels, counts):
    print(f"Cluster {label}: {count} samples ({count/len(kmeans.cluster_labels)*100:.1f}%)")

# Plot cluster sizes
kmeans.plot_cluster_sizes(figsize=(10, 4))

In [ ]:
# Experiment 3: Detailed Silhouette Analysis
print("=== Experiment 3: Silhouette Analysis ===")

# Plot detailed silhouette analysis
kmeans.plot_silhouette_analysis(X_original, figsize=(15, 8))

# Print silhouette statistics
silhouette_metrics = kmeans.metrics
print(f"\nSilhouette Statistics:")
print(f"Average Silhouette Score: {silhouette_metrics.get('silhouette_score', 0):.4f}")
print(f"Positive Silhouette Percentage: {silhouette_metrics.get('positive_silhouette_percentage', 0):.1f}%")
print(f"Balance Score: {silhouette_metrics.get('balance_score', 0):.4f}")

In [ ]:
# Experiment 4: Cluster Centers Analysis
print("=== Experiment 4: Cluster Centers Analysis ===")

# Plot cluster centers heatmap
kmeans.plot_cluster_centers(feature_names=feature_names, figsize=(15, 8))

# Get cluster characteristics
cluster_characteristics = kmeans.get_cluster_characteristics(X_original, feature_names)
print(f"\nCluster Characteristics:")
display(cluster_characteristics[['cluster_id', 'size', 'percentage']].round(2))

# Show top distinguishing features for each cluster
print("\nTop Distinguishing Features:")
for i in range(optimal_k):
    cluster_data = cluster_characteristics[cluster_characteristics['cluster_id'] == i]
    vs_overall_cols = [col for col in cluster_data.columns if col.endswith('_vs_overall')]
    
    if vs_overall_cols:
        top_features = cluster_data[vs_overall_cols].iloc[0].sort_values(key=abs, ascending=False).head(3)
        print(f"\nCluster {i}:")
        for feature, diff in top_features.items():
            feature_name = feature.replace('_vs_overall', '')
            direction = "higher" if diff > 0 else "lower"
            print(f"  - {feature_name}: {direction} than average ({diff:+.3f})")

In [ ]:
# Experiment 5: Interactive K-Means Analysis
print("=== Experiment 5: Interactive Analysis ===")

# Create interactive cluster analysis dashboard
interactive_kmeans_fig = kmeans.create_interactive_cluster_analysis(X_original, feature_names)
interactive_kmeans_fig.show()

# Create interactive cluster plot
if X_pca_2d is not None:
    interactive_plot = kmeans.create_interactive_cluster_plot(
        X_original, method='pca'
    )
    interactive_plot.show()
else:
    print("2D visualization data not available")

## 3. Hierarchical Clustering Experiments

In [ ]:
# Initialize Hierarchical model
hierarchical = HierarchicalModel()

print("=== Hierarchical Clustering Experiments ===")
print(f"Data shape: {X_original.shape}")
print(f"Linkage method: {hierarchical.linkage_method}")
print(f"Distance metric: {hierarchical.metric}")

In [ ]:
# Experiment 1: Compare Linkage Methods
print("=== Experiment 1: Comparing Linkage Methods ===")

# Compare different linkage methods
linkage_results = hierarchical.compare_linkage_methods(
    X_original,
    linkage_methods=['ward', 'complete', 'average'],
    figsize=(15, 10)
)

print("\nLinkage Method Comparison:")
for method, cophenet_corr in linkage_results.items():
    print(f"{method}: {cophenet_corr:.4f}")

# Select best linkage method
best_linkage = max(linkage_results, key=linkage_results.get)
print(f"\nBest linkage method: {best_linkage}")

In [ ]:
# Experiment 2: Find Optimal Number of Clusters
print("=== Experiment 2: Finding Optimal Number of Clusters ===")

# Find optimal number of clusters
optimal_k_hierarchical = hierarchical.find_optimal_clusters(
    X_original,
    k_range=(2, 12),
    method='silhouette',
    plot_results=True
)

print(f"Optimal number of clusters: {optimal_k_hierarchical}")

# Fit hierarchical clustering with optimal k
hierarchical.fit(
    X_original,
    n_clusters=optimal_k_hierarchical,
    linkage_method=best_linkage
)

# Display model summary
hierarchical_summary = hierarchical.get_model_summary()
print(f"\nHierarchical Model Summary:")
for key, value in hierarchical_summary.items():
    if key not in ['linkage_matrix', 'cluster_centers']:
        print(f"{key}: {value}")

# Display cluster distribution
print(f"\nCluster Distribution:")
unique_labels, counts = np.unique(hierarchical.cluster_labels, return_counts=True)
for label, count in zip(unique_labels, counts):
    print(f"Cluster {label}: {count} samples ({count/len(hierarchical.cluster_labels)*100:.1f}%)")

In [ ]:
# Experiment 3: Dendrogram Analysis
print("=== Experiment 3: Dendrogram Analysis ===")

# Plot full dendrogram
hierarchical.plot_dendrogram(
    X_original,
    truncate_mode='lastp',
    p=20,
    figsize=(15, 8)
)

# Create interactive dendrogram
interactive_dendrogram = hierarchical.create_interactive_dendrogram(X_original)
interactive_dendrogram.show()

# Print cophenetic correlation
print(f"\nCophenetic Correlation: {hierarchical.cophenet_corr:.4f}")
print("(Higher values indicate better preservation of original distances)")

In [ ]:
# Experiment 4: Cluster Analysis
print("=== Experiment 4: Hierarchical Cluster Analysis ===")

# Plot cluster centers heatmap
hierarchical.plot_cluster_heatmap(X_original, feature_names, figsize=(15, 8))

# Plot cluster sizes
hierarchical.plot_cluster_sizes(figsize=(10, 4))

# Get cluster statistics
hierarchical_stats = hierarchical.get_cluster_statistics(X_original, feature_names)
print(f"\nHierarchical Cluster Statistics:")
display(hierarchical_stats[['cluster_id', 'size', 'percentage']].round(2))

In [ ]:
# Experiment 5: Interactive Hierarchical Analysis
print("=== Experiment 5: Interactive Hierarchical Analysis ===")

# Create interactive cluster analysis dashboard
interactive_hierarchical_fig = hierarchical.create_interactive_cluster_analysis(X_original, feature_names)
interactive_hierarchical_fig.show()

# Create interactive cluster plot
if X_pca_2d is not None:
    interactive_plot_h = hierarchical.create_interactive_cluster_plot(
        X_original, method='pca'
    )
    interactive_plot_h.show()
else:
    print("2D visualization data not available")

## 4. Model Comparison and Evaluation

In [ ]:
# Evaluate both models comprehensively
print("=== Comprehensive Model Evaluation ===")

# Evaluate K-Means
kmeans_results = evaluator.evaluate_single_model(
    X_original, kmeans.cluster_labels, "K-Means", feature_names
)

# Evaluate Hierarchical
hierarchical_results = evaluator.evaluate_single_model(
    X_original, hierarchical.cluster_labels, "Hierarchical", feature_names
)

print("\nK-Means Evaluation:")
print(f"Overall Score: {kmeans_results['overall_score']:.3f}")
print(f"Silhouette Score: {kmeans_results['internal_metrics']['silhouette_score']:.4f}")
print(f"Davies-Bouldin Score: {kmeans_results['internal_metrics']['davies_bouldin_score']:.4f}")
print(f"Balance Score: {kmeans_results['quality_metrics']['balance_score']:.4f}")

print("\nHierarchical Evaluation:")
print(f"Overall Score: {hierarchical_results['overall_score']:.3f}")
print(f"Silhouette Score: {hierarchical_results['internal_metrics']['silhouette_score']:.4f}")
print(f"Davies-Bouldin Score: {hierarchical_results['internal_metrics']['davies_bouldin_score']:.4f}")
print(f"Balance Score: {hierarchical_results['quality_metrics']['balance_score']:.4f}")

In [ ]:
# Compare models
model_results = {
    "K-Means": kmeans_results,
    "Hierarchical": hierarchical_results
}

comparison_df = evaluator.compare_models(model_results)
print("\n=== Model Comparison ===")
display(comparison_df.round(4))

# Plot comparison
evaluator.plot_model_comparison(comparison_df, figsize=(15, 10))

# Create interactive comparison
interactive_comparison = evaluator.create_interactive_comparison(comparison_df)
interactive_comparison.show()

In [ ]:
# Visualize clustering results side by side
print("=== Clustering Results Visualization ===")

if X_pca_2d is not None:
    fig, axes = plt.subplots(1, 2, figsize=(15, 6))
    
    # K-Means visualization
    scatter1 = axes[0].scatter(X_pca_2d[:, 0], X_pca_2d[:, 1], 
                              c=kmeans.cluster_labels, cmap='viridis', alpha=0.7, s=50)
    axes[0].set_title(f'K-Means Clustering\n({kmeans.n_clusters} clusters)')
    axes[0].set_xlabel('PCA Component 1')
    axes[0].set_ylabel('PCA Component 2')
    axes[0].grid(True, alpha=0.3)
    
    # Hierarchical visualization
    scatter2 = axes[1].scatter(X_pca_2d[:, 0], X_pca_2d[:, 1], 
                              c=hierarchical.cluster_labels, cmap='viridis', alpha=0.7, s=50)
    axes[1].set_title(f'Hierarchical Clustering\n({hierarchical.n_clusters} clusters)')
    axes[1].set_xlabel('PCA Component 1')
    axes[1].set_ylabel('PCA Component 2')
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
else:
    print("2D visualization data not available")
    
    # Create 2D visualization using PCA on the fly
    from sklearn.decomposition import PCA
    pca_viz = PCA(n_components=2, random_state=42)
    X_viz = pca_viz.fit_transform(X_original)
    
    fig, axes = plt.subplots(1, 2, figsize=(15, 6))
    
    # K-Means visualization
    axes[0].scatter(X_viz[:, 0], X_viz[:, 1], 
                   c=kmeans.cluster_labels, cmap='viridis', alpha=0.7, s=50)
    axes[0].set_title(f'K-Means Clustering\n({kmeans.n_clusters} clusters)')
    axes[0].set_xlabel('PCA Component 1')
    axes[0].set_ylabel('PCA Component 2')
    axes[0].grid(True, alpha=0.3)
    
    # Hierarchical visualization
    axes[1].scatter(X_viz[:, 0], X_viz[:, 1], 
                   c=hierarchical.cluster_labels, cmap='viridis', alpha=0.7, s=50)
    axes[1].set_title(f'Hierarchical Clustering\n({hierarchical.n_clusters} clusters)')
    axes[1].set_xlabel('PCA Component 1')
    axes[1].set_ylabel('PCA Component 2')
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

## 5. Business Insights and Cluster Interpretation

In [ ]:
# Generate business insights for the best model
print("=== Business Insights Generation ===")

# Determine best model based on overall score
best_model_name = comparison_df.iloc[0]['Model']
best_score = comparison_df.iloc[0]['Overall Score']

if best_model_name == "K-Means":
    best_model = kmeans
    best_labels = kmeans.cluster_labels
else:
    best_model = hierarchical
    best_labels = hierarchical.cluster_labels

print(f"Best performing model: {best_model_name} (Score: {best_score:.3f})")

# Create cluster profiles
cluster_profiles = []
X_df = pd.DataFrame(X_original, columns=feature_names)
X_df['cluster'] = best_labels

for cluster_id in sorted(np.unique(best_labels)):
    cluster_data = X_df[X_df['cluster'] == cluster_id]
    overall_data = X_df.drop('cluster', axis=1)
    
    profile = {
        'cluster_id': cluster_id,
        'size': len(cluster_data),
        'percentage': (len(cluster_data) / len(X_df)) * 100,
        'characteristics': []
    }
    
    # Find distinguishing features
    for feature in feature_names[:10]:  # Analyze top 10 features
        cluster_mean = cluster_data[feature].mean()
        overall_mean = overall_data[feature].mean()
        overall_std = overall_data[feature].std()
        
        # Calculate z-score difference
        z_diff = (cluster_mean - overall_mean) / overall_std if overall_std > 0 else 0
        
        if abs(z_diff) > 0.5:  # Significant difference
            direction = "high" if z_diff > 0 else "low"
            profile['characteristics'].append(f"{direction} {feature} ({z_diff:+.2f} std)")
    
    cluster_profiles.append(profile)

# Display cluster profiles
print("\n=== Cluster Profiles ===")
for profile in cluster_profiles:
    print(f"\nCluster {profile['cluster_id']} ({profile['size']} customers, {profile['percentage']:.1f}%):")
    for char in profile['characteristics'][:5]:  # Top 5 characteristics
        print(f"  - {char}")

In [ ]:
# Generate business recommendations
print("=== Business Recommendations ===")

recommendations = []

for profile in cluster_profiles:
    cluster_id = profile['cluster_id']
    size = profile['size']
    characteristics = profile['characteristics']
    
    # Generate segment name based on characteristics
    if 'high annual_income' in str(characteristics) and 'high spending_score' in str(characteristics):
        segment_name = "Premium High-Value Customers"
        strategy = "Focus on retention programs, exclusive offers, and premium services"
    elif 'high annual_income' in str(characteristics):
        segment_name = "High-Income Potential"
        strategy = "Upselling opportunities, premium product recommendations"
    elif 'high spending_score' in str(characteristics):
        segment_name = "Frequent Spenders"
        strategy = "Loyalty programs, cross-selling, volume discounts"
    elif 'low annual_income' in str(characteristics) and 'low spending_score' in str(characteristics):
        segment_name = "Budget-Conscious Customers"
        strategy = "Value-focused promotions, discounts, budget-friendly products"
    elif 'low purchase_frequency' in str(characteristics):
        segment_name = "Inactive Customers"
        strategy = "Re-engagement campaigns, special offers, satisfaction surveys"
    else:
        segment_name = f"Segment {cluster_id}"
        strategy = "Further analysis needed for targeted strategy"
    
    recommendations.append({
        'cluster_id': cluster_id,
        'segment_name': segment_name,
        'size': size,
        'strategy': strategy
    })

# Display recommendations
for rec in recommendations:
    print(f"\n{rec['segment_name']} (Cluster {rec['cluster_id']}):")
    print(f"  Size: {rec['size']} customers")
    print(f"  Strategy: {rec['strategy']}")

In [ ]:
# Create final insights report
print("=== Generating Final Report ===")

# Generate comprehensive evaluation report
evaluation_report = evaluator.generate_evaluation_report(
    model_results,
    output_path='../reports/clustering_evaluation_report.md'
)

# Save clustering results
clustering_results = {
    'best_model': best_model_name,
    'best_score': best_score,
    'kmeans_results': {
        'n_clusters': kmeans.n_clusters,
        'overall_score': kmeans_results['overall_score'],
        'metrics': kmeans_results['internal_metrics']
    },
    'hierarchical_results': {
        'n_clusters': hierarchical.n_clusters,
        'overall_score': hierarchical_results['overall_score'],
        'metrics': hierarchical_results['internal_metrics'],
        'cophenet_corr': hierarchical.cophenet_corr
    },
    'cluster_profiles': cluster_profiles,
    'business_recommendations': recommendations
}

save_json(clustering_results, '../reports/clustering_results.json')

# Save models
kmeans.save_model('../models/production/kmeans_model.pkl')
hierarchical.save_model('../models/production/hierarchical_model.pkl')

# Save cluster assignments
results_df = pd.DataFrame(X_original, columns=feature_names)
results_df['kmeans_cluster'] = kmeans.cluster_labels
results_df['hierarchical_cluster'] = hierarchical.cluster_labels
results_df['best_cluster'] = best_labels

results_df.to_csv('../data/processed/cluster_assignments.csv', index=False)

print(f"\n=== CLUSTERING EXPERIMENTS COMPLETED ===")
print(f"Best model: {best_model_name}")
print(f"Overall score: {best_score:.3f}")
print(f"\nFiles saved:")
print(f"- Evaluation report: ../reports/clustering_evaluation_report.md")
print(f"- Clustering results: ../reports/clustering_results.json")
print(f"- K-Means model: ../models/production/kmeans_model.pkl")
print(f"- Hierarchical model: ../models/production/hierarchical_model.pkl")
print(f"- Cluster assignments: ../data/processed/cluster_assignments.csv")
print(f"\nReady to proceed with Phase 5: Evaluation + Insights!")